In [ ]:
# Ô CODE 1 — Thiết lập thử nghiệm và lưới quét thô
import gc
import math
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

try:
    from IPython.display import display
except ImportError:  # Cho phép chạy smoke-test bằng Python thuần ngoài Jupyter.
    display = print

SEED = 43
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = SEED) -> None:
    """Cố định toàn bộ nguồn ngẫu nhiên để phép quét có thể tái lập."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

# Quét từ mức khám phá rất yếu tới rất mạnh; các tham số UCB còn lại được khóa.
BETA_0_LIST = [0.01, 0.03, 0.05, 0.10, 0.15, 0.30, 0.50]
BETA_DECAY = 0.91
DELTA = 1.0

# Cấu hình smoke-test. Không có giá trị nào được ghi ngược vào UCB_2*.py.
EPISODES = 45
GAMMA = 0.95
SARSA_ALPHA = 0.60
Q_LR = 5e-5
Q_WEIGHT_DECAY = 1e-4
VAE_LR = 1e-3
VAE_BETA_KL = 0.01
VAE_BATCH_SIZE = 256
VAE_BOOTSTRAP_UPDATES = 100
VAE_ONLINE_UPDATES = 1
REPLAY_CAPACITY = 50_000
BALANCE_INIT = 1_000.0
TRANSACTION_FEE = 0.001
W_RISK = 0.15
W_STABILITY = 0.05
ACTION_VALUES = np.arange(-5, 6, dtype=np.int64)

print({"seed": SEED, "device": str(DEVICE), "beta_grid": BETA_0_LIST})


In [ ]:
# Ô CODE 2 — Dữ liệu HPG BAD và Frozen Standardization
def find_project_root() -> Path:
    """Tìm root chứa data/ khi chạy ở local, VS Code hoặc Kaggle."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "SARSA_FinancialRL", Path("/kaggle/working/SARSA_FinancialRL"), *cwd.parents]
    for candidate in candidates:
        if (candidate / "data" / "data_storer" / "data_research").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa data/data_storer/data_research.")


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "data_storer" / "data_research"
TRAIN_CSV = DATA_ROOT / "train" / "bad_train_HPG.csv"
TEST_CSV = DATA_ROOT / "test" / "bad_test_HPG.csv"
REQUIRED_COLUMNS = ["time", "close", "MACD", "RSI", "CCI", "ADX"]
STATE_COLUMNS = ["Price", "Balance", "Position", "MACD", "RSI", "CCI", "ADX"]


def load_bad_hpg() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(TRAIN_CSV)
    test = pd.read_csv(TEST_CSV)
    for name, frame in (("train", train), ("test", test)):
        missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
        if missing:
            raise ValueError(f"HPG BAD {name} thiếu cột: {missing}")
        frame["time"] = pd.to_datetime(frame["time"], errors="raise")
        if frame[REQUIRED_COLUMNS[1:]].isna().any().any():
            raise ValueError(f"HPG BAD {name} chứa NaN trong đặc trưng bắt buộc.")
    if train["time"].max() >= test["time"].min():
        raise ValueError("Mốc thời gian train/test chồng lấn; dừng để tránh leakage.")
    return train.reset_index(drop=True), test.reset_index(drop=True)


@dataclass(frozen=True)
class FrozenStandardScaler:
    """Scaler bất biến: chỉ giữ mean/std đã fit từ calibration states của train."""
    mean: np.ndarray
    std: np.ndarray

    def transform(self, states: np.ndarray) -> np.ndarray:
        x = np.asarray(states, dtype=np.float32)
        return (x - self.mean) / self.std


def make_raw_state(row: pd.Series, cash: float, position: int) -> np.ndarray:
    return np.asarray(
        [row["close"], cash, position, row["MACD"], row["RSI"], row["CCI"], row["ADX"]],
        dtype=np.float32,
    )


def training_only_calibration_states(
    train: pd.DataFrame,
    trajectories: int = 5,
    seed: int = SEED,
) -> np.ndarray:
    """Sinh state hiệu chỉnh bằng random policy, tuyệt đối không đọc test.

    Balance và Position là biến động lực, không có sẵn trong CSV. Vì vậy ta chạy
    các trajectory ngẫu nhiên chỉ trên giá train để ước lượng mean/std hợp lệ của
    đủ 7 chiều state. Scaler sau bước này được khóa cho toàn bộ train và test.
    """
    rng = np.random.default_rng(seed)
    collected: List[np.ndarray] = []
    for _ in range(trajectories):
        cash, position = BALANCE_INIT, 0
        for _, row in train.iterrows():
            collected.append(make_raw_state(row, cash, position))
            price = float(row["close"])
            requested = int(rng.choice(ACTION_VALUES))
            if requested > 0:
                affordable = int(cash // (price * (1.0 + TRANSACTION_FEE)))
                executed = min(requested, affordable)
            else:
                executed = -min(-requested, position)
            traded_value = abs(executed) * price
            cash -= executed * price + traded_value * TRANSACTION_FEE
            position += executed
    return np.asarray(collected, dtype=np.float32)


train_hpg, test_hpg = load_bad_hpg()
calibration_states = training_only_calibration_states(train_hpg)
frozen_mean = calibration_states.mean(axis=0, dtype=np.float64).astype(np.float32)
frozen_std = calibration_states.std(axis=0, dtype=np.float64).astype(np.float32)
frozen_std = np.maximum(frozen_std, 1e-6).astype(np.float32)
FROZEN_SCALER = FrozenStandardScaler(mean=frozen_mean.copy(), std=frozen_std.copy())

# Kiểm tra bất biến và xác nhận không có thống kê nào được lấy từ test.
FROZEN_SCALER.mean.setflags(write=False)
FROZEN_SCALER.std.setflags(write=False)
scaler_table = pd.DataFrame({"feature": STATE_COLUMNS, "train_mean": frozen_mean, "train_std": frozen_std})
display(scaler_table)
print({"train_rows": len(train_hpg), "test_rows": len(test_hpg), "train_end": train_hpg.time.max(), "test_start": test_hpg.time.min()})


In [ ]:
# Ô CODE 3 — Q-Network và EpistemicVAE chống quá khớp
class QNetwork(nn.Module):
    """Kiến trúc bị khóa đúng MLP 7 → 32 → 11."""
    def __init__(self) -> None:
        super().__init__()
        self.network = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 11))

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return self.network(state)


class EpistemicVAE(nn.Module):
    """VAE joint (7 state + 11 action one-hot) → latent 16 → 18."""
    def __init__(self, latent_dim: int = 16) -> None:
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(18, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 18))

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        hidden = self.encoder(x)
        return self.fc_mu(hidden), self.fc_logvar(hidden)

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor):
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        return self.decoder(self.reparameterize(mu, logvar)), mu, logvar

    def compute_u_ep(
        self,
        states: torch.Tensor,
        actions_onehot: torch.Tensor,
        delta: float = DELTA,
    ) -> torch.Tensor:
        """Novelty = delta·KL + L2 tại probe biên z⁺ = μ + 1.96σ.

        KL lấy mean theo latent dimension để score không tăng cơ học khi đổi latent_dim.
        L2 dùng trọng số bằng nhau, không có tham số tự do cho từng đặc trưng.
        """
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        kl = -0.5 * torch.mean(1.0 + logvar - mu.square() - logvar.exp(), dim=-1)
        std = torch.exp(0.5 * logvar)
        x_recon_95 = self.decoder(mu + 1.96 * std)
        recon_err = torch.norm(x - x_recon_95, p=2, dim=-1)
        return delta * kl + recon_err


def vae_loss(
    reconstructed: torch.Tensor,
    target: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    beta_kl: float = VAE_BETA_KL,
    robust: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Huber reconstruction + KL với beta_kl mặc định khóa ở 0.01."""
    recon = F.huber_loss(reconstructed, target, reduction="mean") if robust else F.mse_loss(reconstructed, target)
    kl = -0.5 * torch.mean(1.0 + logvar - mu.square() - logvar.exp())
    return recon + beta_kl * kl, recon.detach(), kl.detach()


def actions_to_onehot(action_indices: torch.Tensor) -> torch.Tensor:
    return F.one_hot(action_indices.long(), num_classes=11).to(dtype=torch.float32)


print(QNetwork(), "\n", EpistemicVAE())


In [ ]:
# Ô CODE 4 — Single-Asset MDP với reward shaping rủi ro/ổn định
class SingleAssetTradingEnv:
    """Môi trường long-only; action là số cổ phiếu mua/bán trong [-5, 5]."""
    def __init__(
        self,
        data: pd.DataFrame,
        initial_cash: float = BALANCE_INIT,
        fee_rate: float = TRANSACTION_FEE,
        w_risk: float = W_RISK,
        w_stability: float = W_STABILITY,
        shaped_reward: bool = True,
    ) -> None:
        if len(data) < 2:
            raise ValueError("Môi trường cần ít nhất hai dòng dữ liệu.")
        self.data = data.reset_index(drop=True)
        self.initial_cash = float(initial_cash)
        self.fee_rate = float(fee_rate)
        self.w_risk = float(w_risk)
        self.w_stability = float(w_stability)
        self.shaped_reward = bool(shaped_reward)
        self.reset()

    def reset(self) -> np.ndarray:
        self.index = 0
        self.cash = self.initial_cash
        self.position = 0
        self.peak_value = self.initial_cash
        self.portfolio_history = [self.initial_cash]
        return self._state()

    def _state(self) -> np.ndarray:
        return make_raw_state(self.data.iloc[self.index], self.cash, self.position)

    def _execute(self, requested: int, price: float) -> int:
        if requested > 0:
            affordable = int(self.cash // (price * (1.0 + self.fee_rate)))
            executed = min(int(requested), affordable)
        else:
            executed = -min(-int(requested), self.position)  # Không bán khống.
        traded_value = abs(executed) * price
        self.cash -= executed * price + traded_value * self.fee_rate
        self.position += executed
        return executed

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict[str, float]]:
        current_price = float(self.data.iloc[self.index]["close"])
        previous_value = self.cash + self.position * current_price
        executed = self._execute(int(action), current_price)

        self.index += 1
        next_price = float(self.data.iloc[self.index]["close"])
        portfolio_value = self.cash + self.position * next_price
        r_profit = portfolio_value - previous_value  # Đã gồm phí do cash bị trừ tại _execute.
        self.peak_value = max(self.peak_value, portfolio_value)
        drawdown = (portfolio_value - self.peak_value) / max(self.peak_value, 1e-8)
        daily_return = next_price / max(current_price, 1e-8) - 1.0
        reward = r_profit
        if self.shaped_reward:
            reward -= self.w_risk * abs(drawdown) + self.w_stability * abs(daily_return)

        self.portfolio_history.append(portfolio_value)
        done = self.index >= len(self.data) - 1
        info = {
            "r_profit": float(r_profit),
            "drawdown": float(drawdown),
            "daily_return": float(daily_return),
            "portfolio_value": float(portfolio_value),
            "executed_action": float(executed),
        }
        return self._state(), float(reward), done, info


# Smoke-check bảo toàn miền action và shape state mà không làm thay đổi dữ liệu/scaler.
_env_check = SingleAssetTradingEnv(train_hpg.iloc[:3])
_next_state, _reward, _done, _info = _env_check.step(5)
assert _next_state.shape == (7,) and _env_check.position >= 0
print("Environment smoke-check passed:", _info)


In [ ]:
# Ô CODE 5 — Replay VAE, UCB policy và cập nhật Deep SARSA
class VAEReplayBuffer:
    def __init__(self, capacity: int = REPLAY_CAPACITY) -> None:
        self.capacity = int(capacity)
        self.states: List[np.ndarray] = []
        self.action_indices: List[int] = []

    def add(self, states: Sequence[np.ndarray], action_indices: Sequence[int]) -> None:
        for state, action_idx in zip(states, action_indices):
            self.states.append(np.asarray(state, dtype=np.float32))
            self.action_indices.append(int(action_idx))
        overflow = len(self.states) - self.capacity
        if overflow > 0:
            del self.states[:overflow]
            del self.action_indices[:overflow]

    def sample(self, batch_size: int) -> Tuple[np.ndarray, np.ndarray]:
        size = min(int(batch_size), len(self.states))
        indices = np.random.choice(len(self.states), size=size, replace=False)
        return np.asarray([self.states[i] for i in indices]), np.asarray([self.action_indices[i] for i in indices])

    def __len__(self) -> int:
        return len(self.states)


def ucb_action(
    q_network: QNetwork,
    vae: EpistemicVAE,
    raw_state: np.ndarray,
    beta: float,
    return_raw_novelty: bool = False,
    greedy: bool = False,
) -> Tuple[int, Optional[np.ndarray]]:
    scaled = FROZEN_SCALER.transform(raw_state.reshape(1, -1))
    state_tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
    q_network.eval()
    vae.eval()
    with torch.no_grad():
        q_values = q_network(state_tensor).squeeze(0)
        state_batch = state_tensor.repeat(11, 1)
        action_batch = torch.eye(11, dtype=torch.float32, device=DEVICE)
        novelty_raw = vae.compute_u_ep(state_batch, action_batch, delta=DELTA)
        novelty_normalized = novelty_raw / (novelty_raw.max() + 1e-8)
        scores = q_values if greedy else q_values + float(beta) * novelty_normalized
        action_index = int(torch.argmax(scores).item())
    raw = novelty_raw.detach().cpu().numpy() if return_raw_novelty else None
    return int(ACTION_VALUES[action_index]), raw


def update_vae(
    vae: EpistemicVAE,
    optimizer: torch.optim.Optimizer,
    replay: VAEReplayBuffer,
    beta_kl: float = VAE_BETA_KL,
    robust: bool = True,
) -> Optional[float]:
    if len(replay) == 0:
        return None
    raw_states, action_indices = replay.sample(VAE_BATCH_SIZE)
    states = torch.as_tensor(FROZEN_SCALER.transform(raw_states), dtype=torch.float32, device=DEVICE)
    indices = torch.as_tensor(action_indices, dtype=torch.long, device=DEVICE)
    actions_onehot = actions_to_onehot(indices)
    reconstructed, mu, logvar = vae(states, actions_onehot)
    target = torch.cat([states, actions_onehot], dim=-1)
    loss, _, _ = vae_loss(reconstructed, target, mu, logvar, beta_kl=beta_kl, robust=robust)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(vae.parameters(), max_norm=5.0)
    optimizer.step()
    return float(loss.detach().cpu())


def bootstrap_vae(
    vae: EpistemicVAE,
    optimizer: torch.optim.Optimizer,
    replay: VAEReplayBuffer,
    beta_kl: float,
    robust: bool,
) -> List[float]:
    """Khởi tạo replay bằng random trajectories chỉ trên train rồi warm-up VAE."""
    rng = np.random.default_rng(SEED)
    for _ in range(5):
        env = SingleAssetTradingEnv(train_hpg, shaped_reward=True)
        state = env.reset()
        states, action_indices = [], []
        done = False
        while not done:
            action_idx = int(rng.integers(0, 11))
            states.append(state.copy())
            action_indices.append(action_idx)
            state, _, done, _ = env.step(int(ACTION_VALUES[action_idx]))
        replay.add(states, action_indices)
    losses = []
    for _ in range(VAE_BOOTSTRAP_UPDATES):
        loss = update_vae(vae, optimizer, replay, beta_kl=beta_kl, robust=robust)
        if loss is not None:
            losses.append(loss)
    return losses


def collect_training_episode(env, q_network, vae, beta):
    state = env.reset()
    states, rewards, action_indices, dones = [], [], [], []
    done = False
    while not done:
        action, _ = ucb_action(q_network, vae, state, beta)
        next_state, reward, done, _ = env.step(action)
        states.append(state.copy())
        rewards.append(reward)
        action_indices.append(int(action + 5))
        dones.append(done)
        state = next_state
    # next_action là action on-policy kế tiếp; phần tử cuối bị mask bởi done.
    next_action_indices = action_indices[1:] + [action_indices[-1]]
    next_states = states[1:] + [state.copy()]
    return states, next_states, rewards, action_indices, next_action_indices, dones


def sarsa_update(
    q_network: QNetwork,
    optimizer: torch.optim.Optimizer,
    trajectory,
    robust: bool = True,
    batch_size: int = 128,
) -> List[float]:
    states, next_states, rewards, actions, next_actions, dones = trajectory
    s = torch.as_tensor(FROZEN_SCALER.transform(np.asarray(states)), dtype=torch.float32, device=DEVICE)
    sn = torch.as_tensor(FROZEN_SCALER.transform(np.asarray(next_states)), dtype=torch.float32, device=DEVICE)
    r = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
    a = torch.as_tensor(actions, dtype=torch.long, device=DEVICE)
    an = torch.as_tensor(next_actions, dtype=torch.long, device=DEVICE)
    terminal = torch.as_tensor(dones, dtype=torch.float32, device=DEVICE)
    losses: List[float] = []
    q_network.train()
    for start in range(0, len(states), batch_size):
        sl = slice(start, min(start + batch_size, len(states)))
        current_q = q_network(s[sl]).gather(1, a[sl, None]).squeeze(1)
        with torch.no_grad():
            next_q = q_network(sn[sl]).gather(1, an[sl, None]).squeeze(1)
            td_target = r[sl] + GAMMA * (1.0 - terminal[sl]) * next_q
            target = (1.0 - SARSA_ALPHA) * current_q.detach() + SARSA_ALPHA * td_target
        loss = F.huber_loss(current_q, target) if robust else F.mse_loss(current_q, target)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(q_network.parameters(), max_norm=5.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return losses


In [ ]:
# Ô CODE 6 — Quét beta, đánh giá generalization/VAE và trực quan hóa
def evaluate_agent(q_network, vae, data, beta, collect_novelty=False, previous_row=None):
    # Với test, prepend dòng cuối train để khớp chính xác biên thời gian của baseline.
    evaluation_data = data if previous_row is None else pd.concat([previous_row.to_frame().T, data], ignore_index=True)
    env = SingleAssetTradingEnv(evaluation_data, shaped_reward=True)
    state, done = env.reset(), False
    novelty_values: List[float] = []
    while not done:
        action, raw = ucb_action(q_network, vae, state, beta, return_raw_novelty=collect_novelty, greedy=True)
        if raw is not None:
            novelty_values.extend(raw.tolist())
        state, _, done, _ = env.step(action)
    return np.asarray(env.portfolio_history), np.asarray(novelty_values)


def financial_metrics(portfolio: np.ndarray, data: pd.DataFrame) -> Dict[str, float]:
    final_profit = float(portfolio[-1] - portfolio[0])
    years = max((data.iloc[-1]["time"] - data.iloc[0]["time"]).days / 365.25, 1.0 / 365.25)
    ratio = max(float(portfolio[-1] / max(portfolio[0], 1e-8)), 1e-12)
    arr = (ratio ** (1.0 / years) - 1.0) * 100.0
    returns = np.diff(portfolio) / np.maximum(np.abs(portfolio[:-1]), 1e-8)
    annual_return_pct = np.mean(returns) * 252.0 * 100.0 if len(returns) else 0.0
    volatility_pct = np.std(returns) * np.sqrt(252.0) * 100.0 if len(returns) else 0.0
    sharpe = 0.0 if volatility_pct < 1e-12 else (annual_return_pct - 2.0) / volatility_pct
    peaks = np.maximum.accumulate(portfolio)
    max_drawdown = float(abs(np.min((portfolio - peaks) / np.maximum(peaks, 1e-8))) * 100.0)
    return {"Final Profit": final_profit, "ARR (%)": float(arr), "Sharpe Ratio": float(sharpe), "Max Drawdown (%)": max_drawdown}


def run_one_configuration(beta_0: float, enhanced: bool = True) -> Dict[str, object]:
    """Chạy độc lập một cấu hình; reset seed đảm bảo so sánh beta công bằng."""
    set_seed(SEED)
    q_network = QNetwork().to(DEVICE)
    vae = EpistemicVAE().to(DEVICE)
    q_optimizer = torch.optim.Adam(
        q_network.parameters(), lr=Q_LR, weight_decay=Q_WEIGHT_DECAY if enhanced else 0.0
    )
    vae_optimizer = torch.optim.Adam(vae.parameters(), lr=VAE_LR)
    replay = VAEReplayBuffer()
    beta_kl = VAE_BETA_KL if enhanced else 1e-3
    robust = enhanced
    vae_losses = bootstrap_vae(vae, vae_optimizer, replay, beta_kl=beta_kl, robust=robust)
    train_curve, test_curve, q_losses = [], [], []

    for episode in range(EPISODES):
        beta = max(0.01, float(beta_0) * (BETA_DECAY ** episode))
        train_env = SingleAssetTradingEnv(train_hpg, shaped_reward=enhanced)
        trajectory = collect_training_episode(train_env, q_network, vae, beta)
        replay.add(trajectory[0], trajectory[3])
        q_losses.extend(sarsa_update(q_network, q_optimizer, trajectory, robust=robust))
        for _ in range(VAE_ONLINE_UPDATES):
            loss = update_vae(vae, vae_optimizer, replay, beta_kl=beta_kl, robust=robust)
            if loss is not None:
                vae_losses.append(loss)
        train_portfolio, _ = evaluate_agent(q_network, vae, train_hpg, beta)
        train_curve.append(float(train_portfolio[-1] - BALANCE_INIT))
        test_portfolio, _ = evaluate_agent(q_network, vae, test_hpg, beta, previous_row=train_hpg.iloc[-1])
        test_curve.append(float(test_portfolio[-1] - BALANCE_INIT))

    final_beta = max(0.01, float(beta_0) * (BETA_DECAY ** (EPISODES - 1)))
    test_portfolio, novelty = evaluate_agent(q_network, vae, test_hpg, final_beta, collect_novelty=True, previous_row=train_hpg.iloc[-1])
    metrics = financial_metrics(test_portfolio, test_hpg)
    return {
        "beta_0": float(beta_0), "final_beta": final_beta, **metrics,
        "train_curve": train_curve, "test_curve": test_curve,
        "vae_losses": vae_losses, "q_losses": q_losses, "novelty": novelty,
        "q_network": q_network, "vae": vae, "enhanced": enhanced,
    }


started = time.perf_counter()
screening_runs: List[Dict[str, object]] = []
for grid_index, beta_0 in enumerate(BETA_0_LIST, start=1):
    run = run_one_configuration(beta_0, enhanced=True)
    screening_runs.append(run)
    print(
        f"[{grid_index}/{len(BETA_0_LIST)}] beta_0={beta_0:.2f} | "
        f"profit={run['Final Profit']:.3f} | sharpe={run['Sharpe Ratio']:.4f}"
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame([
    {key: run[key] for key in ["beta_0", "final_beta", "Final Profit", "ARR (%)", "Sharpe Ratio", "Max Drawdown (%)"]}
    for run in screening_runs
]).sort_values("beta_0").reset_index(drop=True)
best_run = max(screening_runs, key=lambda item: (item["Sharpe Ratio"], item["Final Profit"]))

# Ablation legacy cùng beta tốt nhất: bỏ reward shaping/L2, dùng MSE và beta_KL=0.001.
# Đây là đối chứng smoke-test, không tuyên bố ý nghĩa thống kê vì chỉ dùng seed 43.
legacy_run = run_one_configuration(float(best_run["beta_0"]), enhanced=False)
comparison_df = pd.DataFrame([
    {"Variant": "Enhanced", **{k: best_run[k] for k in ["beta_0", "Final Profit", "ARR (%)", "Sharpe Ratio", "Max Drawdown (%)"]}},
    {"Variant": "Legacy ablation", **{k: legacy_run[k] for k in ["beta_0", "Final Profit", "ARR (%)", "Sharpe Ratio", "Max Drawdown (%)"]}},
])

print("\n### Kết quả quét thô (Seed 43)")
print(results_df.to_markdown(index=False, floatfmt=".6f"))
print("\n### Đối chứng tại beta tốt nhất")
print(comparison_df.to_markdown(index=False, floatfmt=".6f"))
print(f"\nBest beta_0={best_run['beta_0']}; tổng thời gian={time.perf_counter() - started:.1f}s")

# Đồ thị 1: sensitivity curve. Hai trục y tránh trộn đơn vị Profit và Sharpe.
fig, ax_profit = plt.subplots(figsize=(10, 5))
ax_sharpe = ax_profit.twinx()
ax_profit.plot(results_df["beta_0"], results_df["Final Profit"], "o-", color="tab:blue", label="Final Profit")
ax_sharpe.plot(results_df["beta_0"], results_df["Sharpe Ratio"], "s--", color="tab:orange", label="Sharpe Ratio")
ax_profit.axvline(best_run["beta_0"], color="grey", alpha=0.5, linestyle=":", label="Best beta")
ax_profit.set(xlabel="BETA_0", ylabel="Final Profit", title="Sensitivity Curve — HPG BAD, Seed 43")
ax_sharpe.set_ylabel("Sharpe Ratio")
ax_profit.grid(alpha=0.25)
fig.legend(loc="upper center", bbox_to_anchor=(0.5, 0.91), ncol=3)
plt.show()

# Đồ thị 2: generalization gap theo episode của cấu hình tốt nhất.
episodes_axis = np.arange(1, EPISODES + 1)
plt.figure(figsize=(10, 5))
plt.plot(episodes_axis, best_run["train_curve"], label="Train final profit")
plt.plot(episodes_axis, best_run["test_curve"], label="Test final profit")
plt.fill_between(episodes_axis, best_run["train_curve"], best_run["test_curve"], alpha=0.12, label="Generalization gap")
plt.xlabel("Episode")
plt.ylabel("Final Profit")
plt.title(f"Generalization Gap — best BETA_0={best_run['beta_0']}")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

# Đồ thị 3: loss VAE và phân phối novelty thô (trước normalize theo max action).
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(best_run["vae_losses"], linewidth=1.2)
axes[0].set(title="VAE Loss", xlabel="VAE update", ylabel="Huber + 0.01·KL")
axes[0].grid(alpha=0.25)
finite_novelty = np.asarray(best_run["novelty"])
finite_novelty = finite_novelty[np.isfinite(finite_novelty)]
axes[1].hist(finite_novelty, bins=40, alpha=0.8, color="tab:green")
axes[1].set(title="Raw Novelty Distribution", xlabel="u_ep raw", ylabel="Frequency")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

if len(finite_novelty):
    novelty_summary = pd.Series(finite_novelty).describe(percentiles=[0.50, 0.90, 0.95, 0.99])
    print("\nNovelty diagnostics:")
    display(novelty_summary.to_frame("u_ep raw"))
    if not np.all(np.isfinite(best_run["vae_losses"])):
        print("CẢNH BÁO: VAE loss xuất hiện NaN/Inf.")
else:
    print("CẢNH BÁO: Không thu được novelty hữu hạn.")

print("\nLưu ý: đây là coarse screening trên duy nhất Seed 43; cần chạy nhiều seed trước khi kết luận thống kê.")
